<a href="https://colab.research.google.com/github/manasabt1997-max/practice-basics/blob/main/3_step_BWA_alignment_(indexing_and_sorting)_%26_convert_SAM_to_BAM_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Read Alignment Demo: BWA + hg38 (small region)

In [16]:
# Step 1: Install required tools
!apt-get install bwa samtools

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bwa is already the newest version (0.7.17-7).
samtools is already the newest version (1.19.2-1build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [17]:
# Step 2: Download a small portion of hg38
!mkdir -p /content/reference
%cd /content/reference
# Clean up any existing files before re-downloading
!rm -f chr22.fa chr22.fa.gz chr22.fa.gz.1
!wget https://hgdownload.soe.ucsc.edu/goldenpath/hg38/chromosomes/chr22.fa.gz --no-check-certificate
!gunzip chr22.fa.gz


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/reference
--2026-09-30 04:16:39--  https://hgdownload.soe.ucsc.edu/goldenpath/hg38/chromosomes/chr22.fa.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12255678 (12M) [application/x-gzip]
Saving to: ‘chr22.fa.gz’

chr22.fa.gz         100%[===================>]  11.69M  15.4MB/s    in 0.8s    

2026-09-30 04:16:41 (15.4 MB/s) - ‘chr22.fa.gz’ saved [12255678/12255678]



In [18]:
# Step 3: Index the reference genome
!bwa index chr22.fa

[bwa_index] Pack FASTA... 0.63 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=101636936, availableWord=19151484
[BWTIncConstructFromPacked] 10 iterations done. 31590664 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 58359704 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 82148056 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 101636936 characters processed.
[bwt_gen] Finished constructing BWT in 40 iterations.
[bwa_index] 84.03 seconds elapse.
[bwa_index] Update BWT... 0.32 sec
[bwa_index] Pack forward-only FASTA... 0.39 sec
[bwa_index] Construct SA from BWT and Occ... 17.84 sec
[main] Version: 0.7.17-r1188
[main] CMD: bwa index chr22.fa
[main] Real time: 104.654 sec; CPU: 103.219 sec


In [15]:
# Step 4: Create a small query FASTQ file with multiple reads
query_seq = """@query1
TTAATTTGGAAGCACACTGACTACTATAATACCAAGAAGTTCGTACGAAGGCAGATAGATAGCCATTAAAATGGCAGAAATTCAACAAAATTTTTAATAG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@query2
GCAGATAGATAGCCATTAAAATGGCAGAAATTCAACAAAATTTTTAATAGTTAATTTGGAAGCACACTGACTACTATAATACCAAGAAGTTCGTACGAAG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
"""
with open("/content/query.fastq", "w") as f:
    f.write(query_seq)


In [19]:
# Step 5: Align the query using BWA-MEM
!bwa mem chr22.fa /content/query.fastq > /content/alignment.sam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 2 sequences (200 bp)...
[M::mem_process_seqs] Processed 2 reads in 0.001 CPU sec, 0.004 real sec
[main] Version: 0.7.17-r1188
[main] CMD: bwa mem chr22.fa /content/query.fastq
[main] Real time: 0.078 sec; CPU: 0.070 sec


In [20]:
# Step 6: View alignment results
!cat /content/alignment.sam

@SQ	SN:chr22	LN:50818468
@PG	ID:bwa	PN:bwa	VN:0.7.17-r1188	CL:bwa mem chr22.fa /content/query.fastq
query1	0	chr22	15524051	60	100M	*	0	0	TTAATTTGGAAGCACACTGACTACTATAATACCAAGAAGTTCGTACGAAGGCAGATAGATAGCCATTAAAATGGCAGAAATTCAACAAAATTTTTAATAG	IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII	NM:i:0	MD:Z:100	AS:i:100	XS:i:0
query2	0	chr22	15524051	60	50S50M	*	0	0	GCAGATAGATAGCCATTAAAATGGCAGAAATTCAACAAAATTTTTAATAGTTAATTTGGAAGCACACTGACTACTATAATACCAAGAAGTTCGTACGAAG	IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII	NM:i:0	MD:Z:50	AS:i:50	XS:i:0	SA:Z:chr22,15524101,+,50M50S,60,0;
query2	2048	chr22	15524101	60	50M50H	*	0	0	GCAGATAGATAGCCATTAAAATGGCAGAAATTCAACAAAATTTTTAATAG	IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII	NM:i:0	MD:Z:50	AS:i:50	XS:i:0	SA:Z:chr22,15524051,+,50S50M,60,0;


Post-Alignment Processing: BAM Conversion, Sorting, and Indexing


In [21]:
# Install SAMtools in the Google Colab environment
# We use apt-get (Ubuntu package manager) since Colab runs on Linux.
!apt-get update
!apt-get install -y samtools
# Verify the installation by checking the version
!samtools --version

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,057 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,637 kB]
Get:14 

SAM to BAM Conversion

In [22]:
# Move back to the main content directory
%cd /content
# samtools view: The tool used to view and convert alignment files.
# -S: Tells the tool the input is a SAM file (legacy option, but good for clarity).
# -b: Tells the tool to output a BAM file.
# The '>' symbol directs the output into a new file called 'unsorted.bam'.
!samtools view -S -b /content/alignment.sam > /content/unsorted.bam
# Let's check the file sizes to see the compression difference!
!ls -lh alignment.sam unsorted.bam

/content
-rw-r--r-- 1 root root 889 Sep 30 04:24 alignment.sam
-rw-r--r-- 1 root root 449 Sep 30 04:27 unsorted.bam


Sorting and Indexing the BAM File

In [23]:
# Sort the BAM file by genomic coordinates.
# -o specifies the output file name.
!samtools sort unsorted.bam -o sorted.bam
# Index the sorted BAM file.
# This will generate a new file named 'sorted.bam.bai' in your folder.
!samtools index sorted.bam
# Finally, let's generate a quick statistical summary of our alignments to ensure everything worked.
!samtools flagstat sorted.bam

3 + 0 in total (QC-passed reads + QC-failed reads)
2 + 0 primary
0 + 0 secondary
1 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
3 + 0 mapped (100.00% : N/A)
2 + 0 primary mapped (100.00% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)
